# 01. Load Reference Synthetic Rockpile

This benchmark reuses the previously generated `Synthetic_Rockpile` dataset instead of creating another pile. The existing pile provides the ground-truth fragment IDs, fragment volumes, labelled point cloud, and PSD/P80 values.

In [ ]:
from pathlib import Path
import sys
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

SOURCE_ROOT = Path(r'C:/Users/creep/code/python/Synthetic_Rockpile')
SOURCE_POINTCLOUD = SOURCE_ROOT / 'outputs' / 'pointclouds' / 'pile_0001_labelled.npz'
SOURCE_METADATA = SOURCE_ROOT / 'outputs' / 'metadata' / 'pile_0001_pointcloud_fragment_metadata.csv'
SOURCE_PSD = SOURCE_ROOT / 'outputs' / 'psd' / 'pile_0001_psd.csv'
SOURCE_SUMMARY = SOURCE_ROOT / 'outputs' / 'metadata' / 'pile_0001_fragmentation_summary.csv'

LABEL_DIR = ROOT / 'data' / 'labels'
SCAN_DIR = ROOT / 'data' / 'scanned_pointclouds'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
for path in [LABEL_DIR, SCAN_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

for path in [SOURCE_POINTCLOUD, SOURCE_METADATA, SOURCE_PSD, SOURCE_SUMMARY]:
    if not path.exists():
        raise FileNotFoundError(path)

source_summary = pd.read_csv(SOURCE_SUMMARY)
source_summary

In [ ]:
cloud = np.load(SOURCE_POINTCLOUD)
points = cloud['points_xyz']
labels = cloud['instance_labels']
metadata = pd.read_csv(SOURCE_METADATA)
gt_psd = pd.read_csv(SOURCE_PSD)
gt_summary = pd.read_csv(SOURCE_SUMMARY)

# Store local benchmark references without duplicating heavy meshes.
np.savez_compressed(
    SCAN_DIR / 'synthetic_rockpile_full_labelled_pointcloud.npz',
    points_xyz=points,
    fragment_ids=labels,
    source_path=str(SOURCE_POINTCLOUD),
)
metadata.to_csv(LABEL_DIR / 'reference_fragments.csv', index=False)
gt_psd.to_csv(TABLE_DIR / 'ground_truth_psd.csv', index=False)
gt_summary.to_csv(TABLE_DIR / 'ground_truth_summary.csv', index=False)

reference_source = pd.DataFrame([{
    'source_repository': str(SOURCE_ROOT),
    'source_pointcloud': str(SOURCE_POINTCLOUD),
    'source_psd': str(SOURCE_PSD),
    'n_points_full_labelled_cloud': int(len(points)),
    'n_ground_truth_fragments': int(gt_summary.loc[0, 'n_fragments']),
    'ground_truth_P50_mm': float(gt_summary.loc[0, 'P50_mm']),
    'ground_truth_P80_mm': float(gt_summary.loc[0, 'P80_mm']),
}])
reference_source.to_csv(TABLE_DIR / 'reference_source_summary.csv', index=False)
reference_source

In [ ]:
rng = np.random.default_rng(2026)
idx = rng.choice(len(points), size=min(65000, len(points)), replace=False)

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points[idx, 0], points[idx, 1], points[idx, 2], c=labels[idx], s=0.8, cmap='tab20')
ax1.set_title('Full labelled Synthetic_Rockpile point cloud')
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]'); ax1.set_zlabel('z [m]')
ax1.view_init(elev=22, azim=-58)

ax2 = fig.add_subplot(122)
diam_col = 'equivalent_diameter_mm' if 'equivalent_diameter_mm' in gt_psd.columns else 'diameter_from_volume_mm'
pass_col = 'cumulative_passing_pct'
ax2.plot(gt_psd[diam_col], gt_psd[pass_col], marker='o', markersize=2.4, linewidth=1.5)
for pct in [50, 80]:
    value = float(gt_summary.loc[0, f'P{pct}_mm'])
    ax2.axhline(pct, linestyle='--', linewidth=1)
    ax2.axvline(value, linestyle='--', linewidth=1, label=f'P{pct} = {value:.1f} mm')
ax2.set_title('Ground-truth PSD from Synthetic_Rockpile')
ax2.set_xlabel('Equivalent spherical diameter [mm]')
ax2.set_ylabel('Cumulative passing [% by volume]')
ax2.grid(alpha=0.25)
ax2.legend()
fig.tight_layout()
figure_path = FIGURE_DIR / 'reference_synthetic_rockpile_and_psd.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)